In [ ]:
import torch
import math
from torch import nn,optim
from torch.nn import functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

data

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    transform=transform,
    download=True,
)
train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=0,
)

utils

In [ ]:
device = torch.device('cuda' if torch.cuda.isavailable() else 'cpu')
num_timesteps = 200
betas = torch.linspace(
    1e-4,
    0.02,
    num_timesteps,
    device=device,
)
alphas = 1.0 - betas
#cumprod是累积乘积
alpha_bars = torch.cumprod(alphas, dim=0)
sqrt_alpha_bars = torch.sqrt(alpha_bars)
sqrt_one_minus_alpha_bars = torch.sqrt(1.0 - alpha_bars)
#返回每个像素对应的sqrtα，形状[64,1,1,1]
def extract(values, timesteps, x_shape):
    """
    根据每个样本的时间步 t，取出对应的扩散参数。

    values: [T]，保存的是每个时间步对应的sqrtα
    timesteps: [batch_size]
    x_shape = [64, 1, 28, 28]
    返回形状: [batch_size, 1, 1, 1]
    """
    batch_size = timesteps.size(0)
    #按照timesteps的数值，从values里边取值
    selected_values = values.gather(0, timesteps)

    return selected_values.view(
        batch_size,
        *((1,) * (len(x_shape) - 1)),
    )
#从x_start图片直接得到xt
def q_sample(x_start, timesteps, noise=None):
    """
    从原始图片 x_start 直接得到任意时间步的带噪图片 x_t。
    """
    if noise is None:
        noise = torch.randn_like(x_start)

    #sqrtα
    sqrt_alpha_bar_t = extract(
        sqrt_alpha_bars,
        timesteps,
        x_start.shape,
    )
    #sqrt(1-α)
    sqrt_one_minus_alpha_bar_t = extract(
        sqrt_one_minus_alpha_bars,
        timesteps,
        x_start.shape,
    )

    x_noisy = (
        sqrt_alpha_bar_t * x_start
        + sqrt_one_minus_alpha_bar_t * noise
    )

    return x_noisy, noise

model

In [ ]:
#对timestep生成对应的sincos位置编码，[batchsize,embedding_dim]
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()
        self.embedding_dim = embedding_dim

    def forward(self, timesteps):
        """
        timesteps: [batch_size]
        return: [batch_size, embedding_dim]
        """
        half_dim = self.embedding_dim // 2
        #生成一系列编码
        frequencies = torch.exp(
            torch.arange(
                half_dim,
                device=timesteps.device,
                dtype=torch.float32,
            )
            * (-math.log(10000) / (half_dim - 1))
        )

        angles = timesteps.float().unsqueeze(1) * frequencies.unsqueeze(0)

        embeddings = torch.cat(
            [torch.sin(angles), torch.cos(angles)],
            dim=1,
        )

        return embeddings
#残差连接，timestep注入，特征提取
class ResBlock(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        time_dim,
    ):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            padding=1,
        )

        self.norm1 = nn.GroupNorm(
            num_groups=8,
            num_channels=out_channels,
        )

        # 把时间编码转换成通道偏置
        self.time_layer = nn.Linear(
            time_dim,
            out_channels,
        )

        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=3,
            padding=1,
        )

        self.norm2 = nn.GroupNorm(
            num_groups=8,
            num_channels=out_channels,
        )

        # 输入、输出通道不同时调整残差
        if in_channels == out_channels:
            self.residual_layer = nn.Identity()
        else:
            self.residual_layer = nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=1,
            )

    def forward(self, x, time_embedding):
        residual = self.residual_layer(x)

        h = self.conv1(x)
        h = self.norm1(h)
        h = F.silu(h)

        # [B, time_dim] -> [B, out_channels, 1, 1]
        time_features = self.time_layer(time_embedding)
        time_features = time_features[:, :, None, None]

        h = h + time_features

        h = self.conv2(h)
        h = self.norm2(h)
        h = F.silu(h)

        return h + residual
class SmallUNet(nn.Module):
    def __init__(
        self,
        image_channels=1,
        base_channels=32,
        time_dim=128,
    ):
        super().__init__()

        # 时间编码网络
        self.time_embedding = nn.Sequential(
            SinusoidalTimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim),
        )

        # 初始卷积：1 × 28 × 28 -> 32 × 28 × 28
        self.input_conv = nn.Conv2d(
            image_channels,
            base_channels,
            kernel_size=3,
            padding=1,
        )

        # --------------------
        # 下采样部分
        # --------------------

        self.down_block1 = ResBlock(
            base_channels,
            base_channels,
            time_dim,
        )

        # 32 × 28 × 28 -> 64 × 14 × 14
        self.downsample1 = nn.Conv2d(
            base_channels,
            base_channels * 2,
            kernel_size=4,
            stride=2,
            padding=1,
        )

        self.down_block2 = ResBlock(
            base_channels * 2,
            base_channels * 2,
            time_dim,
        )

        # 64 × 14 × 14 -> 128 × 7 × 7
        self.downsample2 = nn.Conv2d(
            base_channels * 2,
            base_channels * 4,
            kernel_size=4,
            stride=2,
            padding=1,
        )

        # --------------------
        # 中间部分
        # --------------------

        self.middle_block1 = ResBlock(
            base_channels * 4,
            base_channels * 4,
            time_dim,
        )

        self.middle_block2 = ResBlock(
            base_channels * 4,
            base_channels * 4,
            time_dim,
        )

        # --------------------
        # 上采样部分
        # --------------------

        # 128 × 7 × 7 -> 64 × 14 × 14
        self.upsample1 = nn.ConvTranspose2d(
            base_channels * 4,
            base_channels * 2,
            kernel_size=4,
            stride=2,
            padding=1,
        )

        # 拼接后通道数：64 + 64 = 128
        self.up_block1 = ResBlock(
            base_channels * 4,
            base_channels * 2,
            time_dim,
        )

        # 64 × 14 × 14 -> 32 × 28 × 28
        self.upsample2 = nn.ConvTranspose2d(
            base_channels * 2,
            base_channels,
            kernel_size=4,
            stride=2,
            padding=1,
        )

        # 拼接后通道数：32 + 32 = 64
        self.up_block2 = ResBlock(
            base_channels * 2,
            base_channels,
            time_dim,
        )

        # 输出预测噪声
        self.output_conv = nn.Conv2d(
            base_channels,
            image_channels,
            kernel_size=1,
        )

    def forward(self, x, timesteps):
        time_embedding = self.time_embedding(timesteps)

        # 初始特征
        x = self.input_conv(x)

        # 下采样
        skip1 = self.down_block1(x, time_embedding)

        x = self.downsample1(skip1)
        skip2 = self.down_block2(x, time_embedding)

        x = self.downsample2(skip2)

        # 中间层
        x = self.middle_block1(x, time_embedding)
        x = self.middle_block2(x, time_embedding)

        # 上采样并加入跳跃连接
        x = self.upsample1(x)
        x = torch.cat([x, skip2], dim=1)
        x = self.up_block1(x, time_embedding)

        x = self.upsample2(x)
        x = torch.cat([x, skip1], dim=1)
        x = self.up_block2(x, time_embedding)

        # 不使用 Sigmoid 或 Tanh，因为预测的是高斯噪声
        predicted_noise = self.output_conv(x)

        return predicted_noise
def ddpm_loss(model, x_start, num_timesteps):
    batch_size = x_start.size(0)

    # 1. 为每张图片随机选择时间步
    timesteps = torch.randint(
        low=0,
        high=num_timesteps,
        size=(batch_size,),
        device=x_start.device,
        dtype=torch.long,
    )

    # 2. 生成真实高斯噪声
    real_noise = torch.randn_like(x_start)

    # 3. 得到时间步 t 的带噪图片
    noisy_images, _ = q_sample(
        x_start=x_start,
        timesteps=timesteps,
        noise=real_noise,
    )

    # 4. U-Net 预测加入的噪声
    predicted_noise = model(
        noisy_images,
        timesteps,
    )

    # 5. 计算真实噪声和预测噪声之间的 MSE
    loss = F.mse_loss(
        predicted_noise,
        real_noise,
    )

    return loss

train

In [ ]:
learning_rate = 2e-4
epochs = 20
model = SmallUNet()
optimizer = optim.AdamW(
    model.parameters(),
    lr=learning_rate,
)
def train_one_epoch(
    model,
    train_loader,
    optimizer,
    device,
    num_timesteps,
    epoch,
):
    model.train()

    total_loss = 0.0

    for batch_index, (images, _) in enumerate(train_loader):
        images = images.to(device)

        optimizer.zero_grad(set_to_none=True)

        # 随机选择时间步、加噪、预测噪声并计算 MSE
        loss = ddpm_loss(
            model=model,
            x_start=images,
            num_timesteps=num_timesteps,
        )

        loss.backward()

        # 防止训练初期梯度过大
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0,
        )

        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch} 完成 | "
        f"平均 Loss: {average_loss:.6f}"
    )

    return average_loss
for epoch in range(1, epochs + 1):
    average_loss = train_one_epoch(
        model=model,
        train_loader=train_loader,
        optimizer=optimizer,
        device=device,
        num_timesteps=num_timesteps,
        epoch=epoch,
    )

test

In [ ]:
# alpha_bar_(t-1)
alpha_bars_previous = torch.cat([
    torch.ones(1, device=device),
    alpha_bars[:-1],
])
# q(x_{t-1} | x_t, x_0) 的方差
posterior_variance = (betas* (1.0 - alpha_bars_previous)/ (1.0 - alpha_bars))
sqrt_reciprocal_alphas = torch.sqrt(1.0 / alphas)
#单步去噪
@torch.no_grad()
def p_sample(model, x, timesteps, timestep_index):
    """
    从 x_t 得到 x_(t-1)。
    """
    #返回每个像素对应的β
    beta_t = extract(
        betas,
        timesteps,
        x.shape,
    )
    #返回每个像素对应的1/sqrtαt
    sqrt_reciprocal_alpha_t = extract(
        sqrt_reciprocal_alphas,
        timesteps,
        x.shape,
    )
    #返回每个像素对应sqrt（1-αt）
    sqrt_one_minus_alpha_bar_t = extract(
        sqrt_one_minus_alpha_bars,
        timesteps,
        x.shape,
    )

    # U-Net 预测 x_t 中包含的噪声
    predicted_noise = model(x, timesteps)

    # DDPM 反向扩散均值
    model_mean = sqrt_reciprocal_alpha_t * (
        x
        - beta_t
        * predicted_noise
        / sqrt_one_minus_alpha_bar_t
    )

    # 最后一步直接返回均值，不再添加随机噪声
    if timestep_index == 0:
        return model_mean
    #返回每个像素对应的真实后验方差
    variance_t = extract(
        posterior_variance,
        timesteps,
        x.shape,
    )

    noise = torch.randn_like(x)

    return model_mean + torch.sqrt(variance_t) * noise
#去噪步骤
@torch.no_grad()
def sample_images(
    model,
    num_samples,
    num_timesteps,
    device,
):
    model.eval()

    # x_T：从标准正态分布中采样
    images = torch.randn(
        num_samples,
        1,
        28,
        28,
        device=device,
    )

    # T-1, T-2, ..., 0
    for timestep_index in reversed(range(num_timesteps)):
        timesteps = torch.full(
            size=(num_samples,),
            fill_value=timestep_index,
            device=device,
            dtype=torch.long,
        )

        images = p_sample(
            model=model,
            x=images,
            timesteps=timesteps,
            timestep_index=timestep_index,
        )

        if timestep_index % 20 == 0:
            print(f"反向扩散时间步: {timestep_index}")

    # 将最终图片限制在训练数据范围 [-1, 1]
    images = images.clamp(-1, 1)

    return images
generated_images = sample_images(
    model=model,
    num_samples=16,
    num_timesteps=num_timesteps,
    device=device,
)
# 从 [-1, 1] 转换到 [0, 1]
generated_images = (generated_images.cpu() + 1) / 2
fig, axes = plt.subplots(
    4,
    4,
    figsize=(6, 6),
)
for image, ax in zip(generated_images, axes.flat):
    ax.imshow(
        image.squeeze(0),
        cmap="gray",
        vmin=0,
        vmax=1,
    )
    ax.axis("off")
plt.tight_layout()
plt.show()